# Google Spreadsheets connector

In [ ]:
#@title Google spreadsheets alternate connector

from google.colab import auth
from google.auth import default
try:
  import gspread
except ModuleNotFoundError:
  if 'google.colab' in str(get_ipython()):
    %pip install gspread
  import gspread
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

#Use these functions to read/write

def read_gsheet(spreadsheet_id, sheet_name,first_row_as_column_names=True):
  sht1 = gc.open_by_key(spreadsheet_id)
  worksheet = sht1.worksheet(sheet_name)
  df = pd.DataFrame(worksheet.get_all_values())
  if first_row_as_column_names == True:
    new_header = df.iloc[0]  # Take the first row as the header
    df = df[1:]  # Remove the first row from the dataframe
    df.columns = new_header
  return df


def write_gsheet(spreadsheet_id, sheet_name, df, range=None, clean=True):
  sht1 = gc.open_by_key(spreadsheet_id)
  try:
    worksheet = sht1.worksheet(sheet_name)
  except BaseException:
    worksheet = sht1.add_worksheet(sheet_name,"999","20")
  if clean:
    worksheet.clear()

  values_list = df.values.tolist()
  df=df.fillna('')

  if range == None:
    worksheet.update([df.columns.tolist()]+ df.values.tolist())
  elif range != None:
    worksheet.update(range, [df.columns.tolist()]+ df.values.tolist())

In [ ]:
import pandas as pd

In [ ]:


from __future__ import annotations
from dataclasses import dataclass, field
from itertools import combinations
from typing import Sequence

import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Data import

In [ ]:
#https://docs.google.com/spreadsheets/d/10pjgA_WKI0N6SeAQ9iuWs3MfKAN-KqRAABP77sWH2bo/edit?gid=0#gid=0

input_sheet='10pjgA_WKI0N6SeAQ9iuWs3MfKAN-KqRAABP77sWH2bo'

escudo_df=read_gsheet(input_sheet,'matriz_escudo')

espada_df=read_gsheet(input_sheet,'matriz_espada')

descarte_df=read_gsheet(input_sheet,'matriz_descarte')

matriz_simple=read_gsheet(input_sheet,'Matriz Simple')

In [ ]:
def build_matrices_from_simple(matriz_simple_df, dependencia_suffix=" Dependencia Mapa"):
    dep_cols = [c for c in matriz_simple_df.columns if c.endswith(dependencia_suffix)]
    team2_players = [c[:-len(dependencia_suffix)] for c in dep_cols]
    known_cols = set(team2_players) | set(dep_cols)

    # whatever column is left over (not an opponent score, not a dependency column)
    # is assumed to be the team1 player-label column -- no hardcoded name needed
    leftover = [c for c in matriz_simple_df.columns if c not in known_cols]
    df = matriz_simple_df.set_index(leftover[0]) if len(leftover) == 1 else matriz_simple_df
    if len(leftover) > 1:
        print(f"WARNING: found {len(leftover)} candidate label columns {leftover} -- "
              f"couldn't auto-detect, using default index instead.")

    descarte_df = df[team2_players].apply(pd.to_numeric, errors="coerce")
    dependencia_df = df[dep_cols].apply(pd.to_numeric, errors="coerce")
    dependencia_df.columns = team2_players
    dependencia_df = dependencia_df.fillna(0)

    escudo_df = (descarte_df + dependencia_df).clip(lower=0, upper=20)
    espada_df = (descarte_df - dependencia_df).clip(lower=0, upper=20)
    return escudo_df, espada_df, descarte_df

In [ ]:
escudo_df, espada_df, descarte_df = build_matrices_from_simple(matriz_simple)

# Code

In [ ]:
"""
Warhammer 40k pairing model -- single-file Colab bundle
=========================================================
Paste this WHOLE file into one Colab cell and run it. It merges four
source modules (presentation helpers, the core game-theory model, the
map-dependency imputation helper, and the live interactive session tool)
into one script with no cross-file imports, since code pasted into a
notebook cell has no access to sibling .py files on disk.

Quick start -- full matrices already known
-------------------------------------------
    escudo_df = ...    # Team1-rows x Team2-columns, values in [0, 20]
    espada_df = ...
    descarte_df = ...

    model = build_model(escudo_df, espada_df, descarte_df)

    print_decision_tree(model, s1=..., s2=..., t1=..., t2=...,
                         escudo_df=escudo_df, espada_df=espada_df, descarte_df=descarte_df)
    # OR, live during an actual pairing negotiation (Enter accepts the default):
    run_live_cli(model, escudo_df, espada_df, descarte_df, my_team="team1")

Quick start -- only descarte_df is known (escudo/espada imputed)
--------------------------------------------------------------------
Instead of asking your players to fill in escudo_df and espada_df by
hand, have YOUR team (only) rate how map-dependent each matchup is on a
0 (no effect) to 10 (extremely map-dependent) scale, in a matrix the
same shape as descarte_df:

    map_dependency_df = ...   # same shape as descarte_df, values in [0, 10]
                               # blank/NaN matchups default to 0 (no effect)

    model, escudo_df, espada_df, was_imputed_df = build_model_from_partial_inputs(
        descarte_df, map_dependency_df=map_dependency_df)

    # was_imputed_df is a same-shape boolean matrix -- True wherever a cell
    # was auto-derived rather than hand-entered, useful to display/audit.

This also supports a MIX: pass partially-filled escudo_df/espada_df
(NaN for the matchups you don't know) alongside descarte_df and
map_dependency_df -- only the missing cells get imputed, anything you
did fill in by hand is trusted and left untouched. See
impute_dependent_matrices() below for the exact rule and more detail.

Recommendation tie-breaking
----------------------------
When two options are within DEFAULT_TIE_TOLERANCE points of each other
(0.75 by default) on vs_equilibrium_opponent, the recommendation prefers
whichever has the lower spread (safer/less volatile), rather than
whichever happens to be a fraction of a point ahead. Pass
tie_tolerance=... to shield_decision_report / throw_decision_report /
espada_decision_report to tune this.

Output is rendered as styled, colored tables (green heatmap on expected
outcome, orange on spread/volatility, a bar on the equilibrium mixing
weight, recommended row highlighted). This only looks right in a
notebook environment like Colab -- everything falls back to plain text
automatically if IPython/jinja2 aren't available.
"""

from __future__ import annotations
from dataclasses import dataclass, field
from itertools import combinations
from typing import Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import linprog

try:
    from IPython.display import display, HTML
    _HAS_IPYTHON = True
except ImportError:
    _HAS_IPYTHON = False


# =======================================================================
# PRESENTATION HELPERS (from w40k_display.py)
# =======================================================================

_MATRIX_COLORS = {
    "escudo": "#eaf2ff",
    "espada": "#f5eaff",
    "descarte": "#f0f0f0",
}

_HEADER_STYLE = [
    {"selector": "th", "props": [("background-color", "#1b2a4a"),
                                  ("color", "white"),
                                  ("font-family", "sans-serif"),
                                  ("padding", "4px 10px")]},
    {"selector": "td", "props": [("font-family", "sans-serif"),
                                  ("padding", "4px 10px")]},
]


def section_header(text: str, level: int = 3) -> None:
    """A bold, underlined section title -- marks the start of a new stage
    (SHIELD / THROW / ESPADA / FINAL PAIRINGS)."""
    if not _HAS_IPYTHON:
        print(f"\n{'=' * 70}\n{text}\n{'=' * 70}")
        return
    display(HTML(
        f"<h{level} style='margin:18px 0 6px 0; color:#1b2a4a; "
        f"border-bottom:2px solid #1b2a4a; padding-bottom:4px; "
        f"font-family:sans-serif'>{text}</h{level}>"
    ))


def subheader(text: str) -> None:
    """A smaller label within a section (e.g. 'Prediction for team2')."""
    if not _HAS_IPYTHON:
        print(f"\n-- {text} --")
        return
    display(HTML(
        f"<div style='margin:10px 0 2px 0; font-weight:600; color:#333; "
        f"font-family:sans-serif'>{text}</div>"
    ))


def callout(text: str, kind: str = "info") -> None:
    """A colored callout box: 'success' (green) for recommendations,
    'warning' (amber) for mixed-equilibrium notices, 'info' (blue) for
    predictions/neutral notes."""
    if not _HAS_IPYTHON:
        import re
        print("-> " + re.sub("<[^<]+?>", "", text))
        return
    palette = {
        "info":    ("#e7f3ff", "#0b4a8f", "#b8dcff"),
        "success": ("#e6f7ec", "#1c6b34", "#b6e6c8"),
        "warning": ("#fff8e1", "#8a6100", "#ffe6a3"),
    }
    bg, fg, border = palette.get(kind, palette["info"])
    display(HTML(
        f"<div style='background:{bg}; color:{fg}; border:1px solid {border}; "
        f"padding:8px 12px; border-radius:6px; margin:6px 0; "
        f"font-family:sans-serif'>{text}</div>"
    ))


def insight_list(sentences) -> None:
    """Bullet list of the plain-English 'choosing X instead of Y is worth
    +N points' comparison sentences."""
    if not sentences:
        return
    if not _HAS_IPYTHON:
        for s in sentences:
            print("  " + s)
        return
    items = "".join(f"<li style='margin:2px 0'>{s}</li>" for s in sentences)
    display(HTML(f"<ul style='margin:4px 0 10px 18px; font-family:sans-serif; color:#444'>{items}</ul>"))


def style_decision_report(df: pd.DataFrame):
    """Style a shield/throw/espada decision report:
      - green heatmap on `vs_equilibrium_opponent` (darker = better for you)
      - orange heatmap on `spread` (darker = more volatile/uncertain)
      - an in-cell blue bar on `equilibrium_weight` (visual mixing proportion)
      - the top (recommended) row highlighted and bolded
    Returns a pandas Styler; falls back to the plain DataFrame if styling
    isn't available in this environment.
    """
    fmt = {
        "vs_equilibrium_opponent": "{:.2f}",
        "worst_case": "{:.2f}",
        "best_case": "{:.2f}",
        "spread": "{:.2f}",
        "gap_to_best": "{:+.2f}",
        "equilibrium_weight": "{:.0%}",
    }
    fmt = {k: v for k, v in fmt.items() if k in df.columns}

    def highlight_top(row):
        if row.name == 0:
            return ["background-color: #d4edda; font-weight: 600"] * len(row)
        return [""] * len(row)

    try:
        styler = df.style.format(fmt).apply(highlight_top, axis=1).set_table_styles(_HEADER_STYLE)
        if "vs_equilibrium_opponent" in df.columns:
            styler = styler.background_gradient(subset=["vs_equilibrium_opponent"], cmap="Greens")
        if "spread" in df.columns:
            styler = styler.background_gradient(subset=["spread"], cmap="Oranges")
        if "equilibrium_weight" in df.columns:
            styler = styler.bar(subset=["equilibrium_weight"], color="#5DADE2", vmin=0, vmax=1)
        try:
            styler = styler.hide(axis="index")
        except AttributeError:
            styler = styler.hide_index()
        return styler
    except Exception:
        return df


def display_report(df: pd.DataFrame) -> None:
    """Display a decision report as a styled table (or plain text if
    styling isn't available)."""
    styled = style_decision_report(df)
    if _HAS_IPYTHON:
        display(styled)
    else:
        print(df.to_string(index=False))


def display_mixers(df: pd.DataFrame) -> None:
    """Display just the option/equilibrium_weight columns of a mixed
    equilibrium, for the 'rotate roughly in these proportions' notes."""
    subset = df[["option", "equilibrium_weight"]].reset_index(drop=True)
    display_report(subset)


def display_final_pairings(matches, my_team: str, my_total: float, opp_total: float) -> None:
    """The final 4-matchup scoreboard, color-coded by which of the three
    matrices (escudo/espada/descarte) each matchup was scored from, plus
    a big colored callout with the final tally."""
    rows = []
    for m in matches:
        t1_name, t2_name = m["team1_player"], m["team2_player"]
        mine, theirs = (t1_name, t2_name) if my_team == "team1" else (t2_name, t1_name)
        my_score = m["team1_score"] if my_team == "team1" else 20 - m["team1_score"]
        rows.append({"matchup type": m["matrix"], "you": mine, "opponent": theirs,
                      "your score / 20": my_score})
    df = pd.DataFrame(rows)

    section_header("FINAL PAIRINGS")

    def color_by_matrix(row):
        bg = _MATRIX_COLORS.get(row["matchup type"], "#ffffff")
        return [f"background-color: {bg}"] * len(row)

    try:
        styler = (df.style
                  .apply(color_by_matrix, axis=1)
                  .format({"your score / 20": "{:.0f}"})
                  .set_table_styles(_HEADER_STYLE))
        try:
            styler = styler.hide(axis="index")
        except AttributeError:
            styler = styler.hide_index()
        if _HAS_IPYTHON:
            display(styler)
        else:
            print(df.to_string(index=False))
    except Exception:
        print(df.to_string(index=False))

    winner_kind = "success" if my_total > opp_total else ("warning" if my_total < opp_total else "info")
    callout(f"<b>You: {my_total:.0f} / 80</b> &nbsp;&mdash;&nbsp; Opponent: {opp_total:.0f} / 80",
            kind=winner_kind)


# =======================================================================
# CORE GAME-THEORY MODEL (from w40k_pairing_model.py)
# =======================================================================

# ---------------------------------------------------------------------
# 1. Generic zero-sum game solver (used for every stage below -- any
#    m x n constant-sum game reduces to this).
# ---------------------------------------------------------------------

@dataclass
class ZeroSumSolution:
    value: float                 # game value from the row (Team1) player's POV
    row_strategy: np.ndarray     # optimal mixed strategy for the row player
    col_strategy: np.ndarray     # optimal mixed strategy for the column player


def solve_zero_sum_game(payoff: np.ndarray) -> ZeroSumSolution:
    """Solve an m x n zero-sum game given a payoff matrix from the row
    player's perspective (row player maximizes, column player minimizes).

    Uses the standard LP formulation of matrix-game minimax:
        max_p min_j sum_i p_i * A[i, j]
    solved as an LP, plus the dual LP for the column player's strategy.
    """
    m, n = payoff.shape

    # --- Row player (maximizer) LP ---
    # variables: p_1..p_m, v   ; minimize -v
    c = np.zeros(m + 1)
    c[-1] = -1.0
    # constraints: v - sum_i p_i * A[i, j] <= 0   for each column j
    A_ub = np.hstack([-payoff.T, np.ones((n, 1))])
    b_ub = np.zeros(n)
    A_eq = np.hstack([np.ones((1, m)), np.zeros((1, 1))])
    b_eq = np.array([1.0])
    bounds = [(0, None)] * m + [(None, None)]
    res_row = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                       bounds=bounds, method="highs")
    if not res_row.success:
        raise RuntimeError(f"Row-player LP failed: {res_row.message}")
    p = res_row.x[:m]
    value = res_row.x[-1]

    # --- Column player (minimizer) LP ---
    # variables: q_1..q_n, v ; minimize v
    c2 = np.zeros(n + 1)
    c2[-1] = 1.0
    # constraints: sum_j q_j * A[i, j] - v <= 0   for each row i
    A_ub2 = np.hstack([payoff, -np.ones((m, 1))])
    b_ub2 = np.zeros(m)
    A_eq2 = np.hstack([np.ones((1, n)), np.zeros((1, 1))])
    b_eq2 = np.array([1.0])
    bounds2 = [(0, None)] * n + [(None, None)]
    res_col = linprog(c2, A_ub=A_ub2, b_ub=b_ub2, A_eq=A_eq2, b_eq=b_eq2,
                       bounds=bounds2, method="highs")
    if not res_col.success:
        raise RuntimeError(f"Column-player LP failed: {res_col.message}")
    q = res_col.x[:n]

    return ZeroSumSolution(value=value, row_strategy=p, col_strategy=q)


# ---------------------------------------------------------------------
# 2. Full-match scoring for one FULLY specified outcome (explicit espada
#    picks -- no resolution/argmax happens here, it just adds up the 4
#    resulting matches). Used both as a building block for solving the
#    espada stage below, and directly by the live-session tool once the
#    real, observed picks are known.
# ---------------------------------------------------------------------

def score_combination(s1, s2, throw1: Sequence, throw2: Sequence,
                       j_star, i_star,
                       team1_players: Sequence, team2_players: Sequence,
                       escudo_df: pd.DataFrame, espada_df: pd.DataFrame,
                       descarte_df: pd.DataFrame):
    """Given shields (s1, s2), throw sets (throw1 from Team1, throw2 from
    Team2), and the two shields' ACTUAL espada picks (j_star = the Team2
    player Team1's shield chose to fight; i_star = the Team1 player
    Team2's shield chose to fight), return the total Team1 score plus the
    4 individual matchups. See solve_espada_stage() if you want the
    game-theoretically optimal picks computed for you instead of
    supplying them directly.
    """
    j_left = [j for j in throw2 if j != j_star][0]
    i_left = [i for i in throw1 if i != i_star][0]

    d1 = [p for p in team1_players if p != s1 and p not in throw1][0]
    d2 = [p for p in team2_players if p != s2 and p not in throw2][0]

    matches = [
        {"team1_player": s1, "team2_player": j_star, "matrix": "escudo",
         "team1_score": escudo_df.loc[s1, j_star]},
        {"team1_player": i_star, "team2_player": s2, "matrix": "espada",
         "team1_score": espada_df.loc[i_star, s2]},
        {"team1_player": i_left, "team2_player": j_left, "matrix": "descarte",
         "team1_score": descarte_df.loc[i_left, j_left]},
        {"team1_player": d1, "team2_player": d2, "matrix": "descarte",
         "team1_score": descarte_df.loc[d1, d2]},
    ]
    total_team1_score = sum(m["team1_score"] for m in matches)
    return total_team1_score, matches


def myopic_espada_choice(s1, s2, thrown_at_s1: Sequence, thrown_at_s2: Sequence,
                          escudo_df: pd.DataFrame, espada_df: pd.DataFrame):
    """NOT used by the main pipeline -- kept only for comparison/reference.

    This is the naive way to resolve stage 3: each shield picks whichever
    opponent maximizes/minimizes just THEIR OWN individual match, ignoring
    the fact that whichever candidate they reject also determines who
    that rejected player ends up facing in the descarte-vs-descarte match.
    Since a team's real total score is the SUM of all 4 matches, this
    "greedy" resolution can pick the wrong candidate whenever the shield
    match is close/tied but the resulting descarte match isn't. Use
    solve_espada_stage() instead -- it accounts for both matches at once.
    """
    j_star = max(thrown_at_s2, key=lambda j: escudo_df.loc[s1, j])
    j_left = [j for j in thrown_at_s2 if j != j_star][0]
    i_star = min(thrown_at_s1, key=lambda i: espada_df.loc[i, s2])
    i_left = [i for i in thrown_at_s1 if i != i_star][0]
    return {"j_star": j_star, "j_left": j_left, "i_star": i_star, "i_left": i_left}


# ---------------------------------------------------------------------
# 3. Stage 3 (espada choice), modeled CORRECTLY as a 2x2 simultaneous
#    game. Team1's shield picks which of throw2's 2 candidates to fight;
#    Team2's shield picks which of throw1's 2 candidates to fight. Each
#    pick also determines -- via whoever gets rejected -- who plays in
#    the resulting descarte-vs-descarte match, so the two shields'
#    choices are coupled through that shared leftover match. Solved
#    exactly like every other stage: build the payoff matrix, hand it to
#    solve_zero_sum_game().
# ---------------------------------------------------------------------

def solve_espada_stage(s1, s2, throw1: Sequence, throw2: Sequence,
                        team1_players: Sequence, team2_players: Sequence,
                        escudo_df: pd.DataFrame, espada_df: pd.DataFrame,
                        descarte_df: pd.DataFrame):
    row_options = list(throw2)   # Team1 shield (s1) picks among these
    col_options = list(throw1)   # Team2 shield (s2) picks among these

    payoff = np.zeros((len(row_options), len(col_options)))
    breakdown = {}
    for a, j_star in enumerate(row_options):
        for b, i_star in enumerate(col_options):
            total, matches = score_combination(
                s1, s2, throw1, throw2, j_star, i_star,
                team1_players, team2_players, escudo_df, espada_df, descarte_df)
            payoff[a, b] = total
            breakdown[(j_star, i_star)] = (total, matches)

    solution = solve_zero_sum_game(payoff)
    return {
        "row_options": row_options,
        "col_options": col_options,
        "payoff": payoff,
        "breakdown": breakdown,
        "solution": solution,
    }


# ---------------------------------------------------------------------
# 4. Stage 2 (throw choice): build the 3x3 payoff matrix for fixed shields
#    and solve it as a zero-sum game. Each cell's value now comes from
#    solving the corrected 2x2 espada-stage subgame, not a single
#    deterministic score.
# ---------------------------------------------------------------------

def throw_options(remaining_players: Sequence, n_thrown: int = 2):
    """All ways to choose which `n_thrown` of the remaining players get
    thrown at the opponent's shield. For 4-player teams remaining=3,
    n_thrown=2 -> 3 options."""
    return list(combinations(remaining_players, n_thrown))


def solve_throw_stage(s1, s2, team1_players, team2_players,
                       escudo_df, espada_df, descarte_df):
    remaining1 = [p for p in team1_players if p != s1]
    remaining2 = [p for p in team2_players if p != s2]
    throws1 = throw_options(remaining1)   # 3 options
    throws2 = throw_options(remaining2)   # 3 options

    payoff = np.zeros((len(throws1), len(throws2)))
    breakdown = {}
    for a, t1 in enumerate(throws1):
        for b, t2 in enumerate(throws2):
            espada_details = solve_espada_stage(
                s1, s2, t1, t2, team1_players, team2_players,
                escudo_df, espada_df, descarte_df)
            payoff[a, b] = espada_details["solution"].value
            breakdown[(t1, t2)] = espada_details

    solution = solve_zero_sum_game(payoff)
    return {
        "throws1": throws1,
        "throws2": throws2,
        "payoff": payoff,
        "breakdown": breakdown,
        "solution": solution,
    }


# ---------------------------------------------------------------------
# 5. Stage 1 (shield choice): build the NxN payoff matrix (value of the
#    nested throw-stage zero-sum game) and solve it as a zero-sum game.
# ---------------------------------------------------------------------

@dataclass
class MatchModelResult:
    team1_players: list
    team2_players: list
    shield_payoff: np.ndarray                  # n x n matrix of throw-stage game values
    shield_solution: ZeroSumSolution
    throw_stage_details: dict                  # {(s1, s2): solve_throw_stage(...) output}
    expected_team1_score: float
    expected_team2_score: float

    def summary(self) -> pd.DataFrame:
        df = pd.DataFrame(self.shield_payoff,
                           index=self.team1_players, columns=self.team2_players)
        return df.round(2)

    def optimal_shield_strategy(self):
        p1 = pd.Series(self.shield_solution.row_strategy, index=self.team1_players, name="P(chosen as Team1 shield)")
        p2 = pd.Series(self.shield_solution.col_strategy, index=self.team2_players, name="P(chosen as Team2 shield)")
        return p1.round(4), p2.round(4)


def build_model(escudo_df: pd.DataFrame, espada_df: pd.DataFrame,
                 descarte_df: pd.DataFrame) -> MatchModelResult:
    team1_players = list(escudo_df.index)
    team2_players = list(escudo_df.columns)
    n1, n2 = len(team1_players), len(team2_players)

    shield_payoff = np.zeros((n1, n2))
    throw_stage_details = {}
    for a, s1 in enumerate(team1_players):
        for b, s2 in enumerate(team2_players):
            details = solve_throw_stage(s1, s2, team1_players, team2_players,
                                         escudo_df, espada_df, descarte_df)
            shield_payoff[a, b] = details["solution"].value
            throw_stage_details[(s1, s2)] = details

    shield_solution = solve_zero_sum_game(shield_payoff)

    return MatchModelResult(
        team1_players=team1_players,
        team2_players=team2_players,
        shield_payoff=shield_payoff,
        shield_solution=shield_solution,
        throw_stage_details=throw_stage_details,
        expected_team1_score=shield_solution.value,
        expected_team2_score=80.0 - shield_solution.value,
    )


# ---------------------------------------------------------------------
# 6. Full enumeration (answers the original "every possible combination"
#    request) -- useful for auditing / sanity-checking the equilibrium.
# ---------------------------------------------------------------------

def enumerate_all_combinations(escudo_df, espada_df, descarte_df) -> pd.DataFrame:
    """Brute-force every (shield1, shield2, throw1, throw2) combination. The
    reported score is now the EQUILIBRIUM value of the nested espada-stage
    subgame for that combination (both shields playing their optimal 2x2
    mix), not a single deterministic outcome -- see solve_espada_stage()
    for the full breakdown of any specific combination.
    """
    team1_players = list(escudo_df.index)
    team2_players = list(escudo_df.columns)
    rows = []
    for s1 in team1_players:
        for s2 in team2_players:
            remaining1 = [p for p in team1_players if p != s1]
            remaining2 = [p for p in team2_players if p != s2]
            for t1 in throw_options(remaining1):
                for t2 in throw_options(remaining2):
                    espada_details = solve_espada_stage(
                        s1, s2, t1, t2, team1_players, team2_players,
                        escudo_df, espada_df, descarte_df)
                    total = espada_details["solution"].value
                    rows.append({
                        "shield1": s1, "shield2": s2,
                        "throw1": t1, "throw2": t2,
                        "team1_expected_score": total,
                        "team2_expected_score": 80 - total,
                    })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# 7. Decision-tree style insights: how much is each option at each
#    decision node actually worth, examined independently of the rest
#    of the tree.
# ---------------------------------------------------------------------

DEFAULT_TIE_TOLERANCE = 0.75  # points; see _option_report's tie-breaking rule below


def _option_report(payoff_for_me: np.ndarray, opp_equilibrium_strategy: np.ndarray,
                    my_equilibrium_strategy: np.ndarray, option_labels: Sequence,
                    tie_tolerance: float = DEFAULT_TIE_TOLERANCE) -> pd.DataFrame:
    """Given a payoff matrix already oriented so rows = my options, columns =
    opponent options, and values = MY score (higher is better for me),
    report the expected outcome for each option assuming the opponent
    sticks to their game-theoretically optimal mix, regardless of what I
    pick (`vs_equilibrium_opponent`). This is the most defensible
    "expected value" number -- it equals the overall game value for
    whichever option(s) are actually part of my own equilibrium mix.

    `worst_case`/`best_case` are the SPREAD around that average: the real
    opponent doesn't actually play a probabilistic blend in any one match,
    they commit to one specific option, so the real result of picking this
    option could land anywhere from `worst_case` (if they happen to pick
    their best counter to you) to `best_case` (if they happen to pick
    whatever helps you most) -- `vs_equilibrium_opponent` is the average
    across those outcomes if this exact situation repeated many times
    against an opponent playing their equilibrium mix, not a guarantee
    about this one match. `spread` = best_case - worst_case, a quick sense
    of how much a single match could bounce around that average.

    `my_equilibrium_strategy` is included so you can see which options the
    solver actually recommends mixing between (weight > 0) versus options
    that are strictly worse in equilibrium (weight == 0).

    Tie-breaking: with real data, two options are rarely EXACTLY equal on
    `vs_equilibrium_opponent` -- they're often off by a fraction of a
    point for reasons that don't really matter (rounding, near-symmetric
    matchups). Ranking purely by that value would recommend whichever one
    happens to be a hair ahead, ignoring that it might also be far riskier.
    So: any option within `tie_tolerance` points of the best value is
    treated as "effectively tied" with it, and among that near-best group,
    the recommendation (row 0) is whichever has the LOWEST spread rather
    than the highest raw value. Options clearly outside that tolerance are
    still ranked purely by `vs_equilibrium_opponent`, below the near-best
    group.
    """
    vs_equilibrium_opp = payoff_for_me @ opp_equilibrium_strategy
    worst_case = payoff_for_me.min(axis=1)
    best_case = payoff_for_me.max(axis=1)

    df = pd.DataFrame({
        "option": list(option_labels),
        "vs_equilibrium_opponent": vs_equilibrium_opp,
        "worst_case": worst_case,
        "best_case": best_case,
        "spread": best_case - worst_case,
        "equilibrium_weight": np.round(my_equilibrium_strategy, 4),
    })
    df["gap_to_best"] = (df["vs_equilibrium_opponent"] - df["vs_equilibrium_opponent"].max()).round(2)

    best_value = df["vs_equilibrium_opponent"].max()
    near_best_mask = (best_value - df["vs_equilibrium_opponent"]) <= tie_tolerance
    near_best = df[near_best_mask].sort_values("spread", ascending=True)
    rest = df[~near_best_mask].sort_values("vs_equilibrium_opponent", ascending=False)
    return pd.concat([near_best, rest]).reset_index(drop=True).round(2)


def shield_decision_report(model: MatchModelResult, perspective: str = "team1",
                            tie_tolerance: float = DEFAULT_TIE_TOLERANCE) -> pd.DataFrame:
    """Stage-1 (shield choice) insight table, examined independently of the
    throw stage underneath it. `perspective` is "team1" or "team2".
    `tie_tolerance` controls the near-tie/lower-spread rule in _option_report.
    """
    if perspective == "team1":
        payoff_for_me = model.shield_payoff                # rows=team1, cols=team2, Team1 score
        opp_strategy = model.shield_solution.col_strategy   # Team2's equilibrium shield mix
        my_strategy = model.shield_solution.row_strategy
        option_labels = model.team1_players
    elif perspective == "team2":
        payoff_for_me = (80.0 - model.shield_payoff).T       # rows=team2, cols=team1, Team2 score
        opp_strategy = model.shield_solution.row_strategy    # Team1's equilibrium shield mix
        my_strategy = model.shield_solution.col_strategy
        option_labels = model.team2_players
    else:
        raise ValueError("perspective must be 'team1' or 'team2'")

    return _option_report(payoff_for_me, opp_strategy, my_strategy, option_labels, tie_tolerance)


def throw_decision_report(model: MatchModelResult, s1, s2, perspective: str = "team1",
                           tie_tolerance: float = DEFAULT_TIE_TOLERANCE) -> pd.DataFrame:
    """Stage-2 (throw choice) insight table for one specific, already-known
    pair of shields (s1 from Team1, s2 from Team2) -- by the time throws
    are chosen, shields are public, so this is conditioned on a real pair
    rather than averaged over Team2's shield mix. `perspective` is
    "team1" or "team2".
    """
    details = model.throw_stage_details[(s1, s2)]
    payoff = details["payoff"]           # rows=team1 throws, cols=team2 throws, Team1 score
    solution = details["solution"]

    if perspective == "team1":
        payoff_for_me = payoff
        opp_strategy = solution.col_strategy
        my_strategy = solution.row_strategy
        option_labels = details["throws1"]
    elif perspective == "team2":
        payoff_for_me = (80.0 - payoff).T
        opp_strategy = solution.row_strategy
        my_strategy = solution.col_strategy
        option_labels = details["throws2"]
    else:
        raise ValueError("perspective must be 'team1' or 'team2'")

    return _option_report(payoff_for_me, opp_strategy, my_strategy, option_labels, tie_tolerance)


def espada_decision_report(s1, s2, throw1: Sequence, throw2: Sequence,
                            team1_players: Sequence, team2_players: Sequence,
                            escudo_df: pd.DataFrame, espada_df: pd.DataFrame,
                            descarte_df: pd.DataFrame, perspective: str = "team1",
                            tie_tolerance: float = DEFAULT_TIE_TOLERANCE) -> pd.DataFrame:
    """Stage-3 (espada choice) insight table for one specific, already-known
    combination of shields AND throws. A shield's pick also determines who
    ends up in the resulting descarte-vs-descarte match, so this is solved
    as a small 2x2 game (see solve_espada_stage) rather than a simple
    best-response. `perspective` is "team1" or "team2".
    """
    details = solve_espada_stage(s1, s2, throw1, throw2, team1_players, team2_players,
                                  escudo_df, espada_df, descarte_df)
    payoff = details["payoff"]     # rows = team1 shield's options (from throw2), cols = team2 shield's options (from throw1)
    solution = details["solution"]

    if perspective == "team1":
        payoff_for_me = payoff
        opp_strategy = solution.col_strategy
        my_strategy = solution.row_strategy
        option_labels = details["row_options"]
    elif perspective == "team2":
        payoff_for_me = (80.0 - payoff).T
        opp_strategy = solution.row_strategy
        my_strategy = solution.col_strategy
        option_labels = details["col_options"]
    else:
        raise ValueError("perspective must be 'team1' or 'team2'")

    return _option_report(payoff_for_me, opp_strategy, my_strategy, option_labels, tie_tolerance)


def insight_sentences(report: pd.DataFrame, metric: str = "vs_equilibrium_opponent",
                       option_col: str = "option") -> list:
    """Turn a decision report into plain-English comparison sentences, e.g.
    'Choosing A1 instead of A3 is worth +10.40 points on vs_equilibrium_opponent.'
    Always compares every option to the top-ranked (recommended) one.

    Because the top row can now be picked for having a lower `spread`
    rather than the strictly highest `vs_equilibrium_opponent` (see the
    tie-breaking rule in _option_report), the gap can occasionally come
    out negative -- that just means the recommended pick is marginally
    lower on this metric but was preferred for being less volatile, and
    is reported as-is (with a correctly-signed "-", not "+-").
    """
    best_label = report.iloc[0][option_col]
    best_value = report.iloc[0][metric]
    sentences = []
    for _, row in report.iterrows():
        if row[option_col] == best_label:
            continue
        gap = best_value - row[metric]
        sentences.append(
            f"Choosing {best_label} instead of {row[option_col]} is worth "
            f"{gap:+.2f} points on '{metric}'."
        )
    return sentences


def print_decision_tree(model: MatchModelResult, s1=None, s2=None, t1=None, t2=None,
                         escudo_df: Optional[pd.DataFrame] = None,
                         espada_df: Optional[pd.DataFrame] = None,
                         descarte_df: Optional[pd.DataFrame] = None):
    """Print the stage-1 shield decision independently; if a specific
    (s1, s2) shield pair is supplied, also the stage-2 throw decision
    nested underneath it; and if specific throws (t1, t2) are supplied
    too (plus the three raw matrices, needed for the stage-3 solve),
    also the stage-3 espada decision. This mirrors the real decision
    sequence: shields are chosen before either side sees the other's
    pick; throws only after both shields are known; espada picks only
    after both throws are known.
    """
    section_header("STAGE 1 &middot; SHIELD (decided before either team sees the other's pick)")
    for team in ("team1", "team2"):
        subheader(f"{team}")
        report = shield_decision_report(model, perspective=team)
        display_report(report)
        insight_list(insight_sentences(report))

    if s1 is not None and s2 is not None:
        section_header(f"STAGE 2 &middot; THROW, GIVEN shields are already known: "
                        f"Team1={s1}, Team2={s2}")
        for team in ("team1", "team2"):
            subheader(f"{team}")
            report = throw_decision_report(model, s1, s2, perspective=team)
            display_report(report)
            insight_list(insight_sentences(report))

    if t1 is not None and t2 is not None and escudo_df is not None:
        section_header(f"STAGE 3 &middot; ESPADA, GIVEN shields and throws are known: "
                        f"Team1 threw {t1}, Team2 threw {t2}")
        subheader("(accounts for the resulting descarte-vs-descarte match, not just each "
                  "shield's own duel)")
        for team in ("team1", "team2"):
            subheader(f"{team}")
            report = espada_decision_report(s1, s2, t1, t2, model.team1_players, model.team2_players,
                                             escudo_df, espada_df, descarte_df, perspective=team)
            display_report(report)
            insight_list(insight_sentences(report))


# =======================================================================
# MAP-DEPENDENCY IMPUTATION (from w40k_imputation.py)
# =======================================================================

DEFAULT_MAP_DEPENDENCY = 0.0
MIN_DEPENDENCY, MAX_DEPENDENCY = 0.0, 10.0
MIN_SCORE, MAX_SCORE = 0.0, 20.0


def _validate_dependency_df(map_dependency_df: pd.DataFrame, descarte_df: pd.DataFrame) -> pd.DataFrame:
    """Reindex onto descarte_df's exact shape (missing rows/cols/cells -> NaN,
    which later becomes the default of 0), and warn about any out-of-range
    values instead of silently misbehaving."""
    aligned = map_dependency_df.reindex(index=descarte_df.index, columns=descarte_df.columns)
    out_of_range = aligned[(aligned < MIN_DEPENDENCY) | (aligned > MAX_DEPENDENCY)]
    if out_of_range.notna().to_numpy().any():
        bad_cells = [(i, j, aligned.loc[i, j]) for i in aligned.index for j in aligned.columns
                     if pd.notna(aligned.loc[i, j]) and not (MIN_DEPENDENCY <= aligned.loc[i, j] <= MAX_DEPENDENCY)]
        raise ValueError(
            f"map_dependency_df has {len(bad_cells)} value(s) outside the valid "
            f"[{MIN_DEPENDENCY}, {MAX_DEPENDENCY}] range, e.g. {bad_cells[:5]}. "
            f"Fix these (or leave them blank/NaN to default to {DEFAULT_MAP_DEPENDENCY})."
        )
    return aligned


def impute_dependent_matrices(
    descarte_df: pd.DataFrame,
    map_dependency_df: Optional[pd.DataFrame] = None,
    escudo_df: Optional[pd.DataFrame] = None,
    espada_df: Optional[pd.DataFrame] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Fill in any missing escudo_df/espada_df cells from descarte_df plus a
    per-matchup map-dependency value, leaving any hand-filled cells as-is.

    Parameters
    ----------
    descarte_df : required, fully filled Team1-rows x Team2-columns matrix
        of baseline (no-map-advantage) scores in [0, 20].
    map_dependency_df : optional, same shape/index/columns as descarte_df,
        values in [0, 10]. Any missing matchup (whole matrix is None, or
        individual cells are NaN, or rows/columns are simply absent)
        defaults to 0 -- i.e. "the map has no effect on this matchup",
        which makes the imputed escudo/espada cell identical to descarte.
    escudo_df, espada_df : optional, same shape as descarte_df. Pass None
        to fully derive from descarte_df + map_dependency_df. Pass a
        partially filled DataFrame (NaN for the matchups you don't know)
        to have ONLY those specific cells imputed -- anything you did
        fill in is trusted and left untouched.

    Returns
    -------
    (escudo_df_filled, espada_df_filled, was_imputed_df) -- the first two
    are fully populated (no NaNs), clipped to [0, 20]. `was_imputed_df`
    is a same-shape boolean DataFrame, True wherever EITHER escudo or
    espada was auto-filled for that matchup, so you can audit at a
    glance how much of the model is resting on the map-dependency
    assumption versus real hand-entered data.
    """
    if descarte_df.isna().to_numpy().any():
        raise ValueError("descarte_df must be fully filled in -- it has no fallback/default of its own.")

    if map_dependency_df is None:
        dependency = pd.DataFrame(np.nan, index=descarte_df.index, columns=descarte_df.columns)
    else:
        dependency = _validate_dependency_df(map_dependency_df, descarte_df)
    dependency = dependency.fillna(DEFAULT_MAP_DEPENDENCY)

    if escudo_df is None:
        escudo_df = pd.DataFrame(np.nan, index=descarte_df.index, columns=descarte_df.columns)
    else:
        escudo_df = escudo_df.reindex(index=descarte_df.index, columns=descarte_df.columns)

    if espada_df is None:
        espada_df = pd.DataFrame(np.nan, index=descarte_df.index, columns=descarte_df.columns)
    else:
        espada_df = espada_df.reindex(index=descarte_df.index, columns=descarte_df.columns)

    escudo_missing = escudo_df.isna()
    espada_missing = espada_df.isna()

    escudo_imputed_values = (descarte_df + dependency).clip(lower=MIN_SCORE, upper=MAX_SCORE)
    espada_imputed_values = (descarte_df - dependency).clip(lower=MIN_SCORE, upper=MAX_SCORE)

    escudo_filled = escudo_df.where(~escudo_missing, escudo_imputed_values)
    espada_filled = espada_df.where(~espada_missing, espada_imputed_values)

    was_imputed_df = escudo_missing | espada_missing

    return escudo_filled, espada_filled, was_imputed_df


def build_matrices_from_simple_format(
    escudo_df: Optional[pd.DataFrame],
    espada_df: Optional[pd.DataFrame],
    descarte_df: Optional[pd.DataFrame],
    matriz_simple_df: pd.DataFrame,
    dependencia_suffix: str = " Dependencia Mapa",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Bridge for the "Matriz Simple" input format: one row per Team1
    player, and for each Team2 opponent `p` two columns -- `p` (the
    general/default score, equivalent to a descarte_df cell) and
    `f"{p}{dependencia_suffix}"` (that matchup's map-dependency, 0-10).

    Any of escudo_df/espada_df/descarte_df can be None or partially
    filled -- real cells always win; descarte_df falls back to the
    matriz_simple general column, and escudo_df/espada_df fall back to
    descarte (+/-) that matchup's dependency, via impute_dependent_matrices.

    Returns (escudo_filled, espada_filled, descarte_filled, was_imputed_df).
    """
    team2_players = [c for c in matriz_simple_df.columns if not c.endswith(dependencia_suffix)]
    team1_players = matriz_simple_df.index

    general_df = matriz_simple_df[team2_players].reindex(index=team1_players, columns=team2_players)
    dependency_df = matriz_simple_df[[f"{p}{dependencia_suffix}" for p in team2_players]].copy()
    dependency_df.columns = team2_players
    dependency_df = dependency_df.reindex(index=team1_players, columns=team2_players)

    if descarte_df is None:
        descarte_df = pd.DataFrame(np.nan, index=team1_players, columns=team2_players)
    descarte_df = descarte_df.reindex(index=team1_players, columns=team2_players)
    descarte_filled = descarte_df.where(~descarte_df.isna(), general_df)

    escudo_filled, espada_filled, was_imputed_df = impute_dependent_matrices(
        descarte_filled, map_dependency_df=dependency_df, escudo_df=escudo_df, espada_df=espada_df)

    return escudo_filled, espada_filled, descarte_filled, was_imputed_df


def build_model_from_partial_inputs(
    descarte_df: pd.DataFrame,
    map_dependency_df: Optional[pd.DataFrame] = None,
    escudo_df: Optional[pd.DataFrame] = None,
    espada_df: Optional[pd.DataFrame] = None,
):
    """Convenience one-liner for Colab: impute whatever's missing, then
    build the full model in one call. Returns (model, escudo_df_filled,
    espada_df_filled, was_imputed_df) -- the filled matrices and the
    imputation mask are returned too so you can display/audit them
    (e.g. via w40k_display.display_report or a plain .style.applymap
    highlight on was_imputed_df) alongside the model results.
    """
    escudo_filled, espada_filled, was_imputed_df = impute_dependent_matrices(
        descarte_df, map_dependency_df=map_dependency_df,
        escudo_df=escudo_df, espada_df=espada_df)
    model = build_model(escudo_filled, espada_filled, descarte_df)
    return model, escudo_filled, espada_filled, was_imputed_df


# =======================================================================
# LIVE PAIRING ASSISTANT (from live_pairing_session.py)
# =======================================================================

class LivePairingSession:
    """Stateful walk-through of one real match's pairing process, from your
    team's point of view. `my_team` is "team1" or "team2" depending on
    which side of the escudo/espada/descarte matrices your team is."""

    def __init__(self, model: MatchModelResult, escudo_df: pd.DataFrame,
                 espada_df: pd.DataFrame, descarte_df: pd.DataFrame,
                 my_team: str = "team1"):
        if my_team not in ("team1", "team2"):
            raise ValueError("my_team must be 'team1' or 'team2'")
        self.model = model
        self.escudo_df, self.espada_df, self.descarte_df = escudo_df, espada_df, descarte_df
        self.my_team = my_team
        self.opp_team = "team2" if my_team == "team1" else "team1"
        self.my_players = model.team1_players if my_team == "team1" else model.team2_players
        self.opp_players = model.team2_players if my_team == "team1" else model.team1_players

        self.my_shield: Optional[str] = None
        self.opp_shield: Optional[str] = None
        self.my_throw: Optional[Tuple[str, str]] = None
        self.opp_throw: Optional[Tuple[str, str]] = None
        self._shield_reports: Optional[Tuple[pd.DataFrame, pd.DataFrame]] = None
        self._throw_reports: Optional[Tuple[pd.DataFrame, pd.DataFrame]] = None
        self._espada_reports: Optional[Tuple[pd.DataFrame, pd.DataFrame]] = None
        self.result: Optional[dict] = None

    # ---- orientation helpers ----------------------------------------
    def _to_team1_team2(self, mine, theirs):
        """Reorder (mine, theirs) into (team1_value, team2_value)."""
        return (mine, theirs) if self.my_team == "team1" else (theirs, mine)

    def _validate_player(self, name, valid_list, field_name):
        if name not in valid_list:
            raise ValueError(f"{field_name}={name!r} is not one of {valid_list}")

    def _validate_pair(self, pair, valid_list, field_name):
        if len(pair) != 2 or len(set(pair)) != 2:
            raise ValueError(f"{field_name} must be exactly 2 distinct players, got {pair}")
        for p in pair:
            if p not in valid_list:
                raise ValueError(f"{field_name} contains {p!r}, which is not one of {valid_list}")

    # ---- Stage 1: shield ----------------------------------------------
    def recommend_shield(self):
        """Call BEFORE you commit to a shield -- the opponent's shield is
        not knowable yet (simultaneous/hidden), so this ranks your options
        against their equilibrium shield mix, and also shows the mirror
        image: what the opponent's own best shield is expected to be.
        Returns (my_top_option, opp_top_option) for use as CLI defaults.
        """
        my_report = shield_decision_report(self.model, perspective=self.my_team)
        opp_report = shield_decision_report(self.model, perspective=self.opp_team)
        self._shield_reports = (my_report, opp_report)

        section_header(f"STAGE 1 &middot; SHIELD &mdash; {self.my_team.upper()}'s choice "
                        f"(opponent's shield not yet known)")
        display_report(my_report)
        insight_list(insight_sentences(my_report))
        my_top = my_report.iloc[0]
        callout(f"Recommended shield: <b>{my_top['option']}</b> "
                f"(equilibrium weight {my_top['equilibrium_weight']:.0%})", kind="success")
        my_mixers = my_report[my_report["equilibrium_weight"] > 0]
        if len(my_mixers) > 1:
            callout("Equilibrium actually MIXES across these shields -- there's no single "
                    "always-correct pick; rotate across matches roughly in these proportions:",
                    kind="warning")
            display_mixers(my_mixers)

        subheader(f"Prediction for {self.opp_team}'s best shield "
                  f"(their mirror-image decision, before they see yours either)")
        display_report(opp_report)
        opp_top = opp_report.iloc[0]
        callout(f"Likely pick if they play rationally: <b>{opp_top['option']}</b> "
                f"(equilibrium weight {opp_top['equilibrium_weight']:.0%}). Confirm once "
                f"revealed -- real opponents don't always play optimally.", kind="info")
        opp_mixers = opp_report[opp_report["equilibrium_weight"] > 0]
        if len(opp_mixers) > 1:
            callout("Their equilibrium also mixes -- don't be surprised by any of:", kind="warning")
            display_mixers(opp_mixers)

        return my_top["option"], opp_top["option"]

    def lock_shields(self, my_shield: str, opp_shield: str):
        """Call once both shields are revealed face-up. Immediately runs the
        stage-2 (throw) recommendation for you, since that's the next real
        decision point."""
        self._validate_player(my_shield, self.my_players, "my_shield")
        self._validate_player(opp_shield, self.opp_players, "opp_shield")
        self.my_shield, self.opp_shield = my_shield, opp_shield
        callout(f"Shields locked &mdash; {self.my_team}: <b>{my_shield}</b> "
                f"| {self.opp_team}: <b>{opp_shield}</b>", kind="info")
        return self.recommend_throw()

    # ---- Stage 2: throw -------------------------------------------------
    def recommend_throw(self):
        """Call once both shields are known: which 2 of your remaining
        players should you throw at the opponent's shield? Also shows what
        the opponent's own best throw is expected to be, now that shields
        are public for both sides. Returns (my_top_option, opp_top_option)
        for use as CLI defaults.
        """
        if self.my_shield is None or self.opp_shield is None:
            raise RuntimeError("Call lock_shields(...) first.")
        s1, s2 = self._to_team1_team2(self.my_shield, self.opp_shield)

        my_report = throw_decision_report(self.model, s1, s2, perspective=self.my_team)
        opp_report = throw_decision_report(self.model, s1, s2, perspective=self.opp_team)
        self._throw_reports = (my_report, opp_report)

        section_header(f"STAGE 2 &middot; THROW &mdash; {self.my_team.upper()}'s choice "
                        f"(vs {self.opp_team} shield {self.opp_shield})")
        display_report(my_report)
        insight_list(insight_sentences(my_report))
        my_top = my_report.iloc[0]
        callout(f"Recommended throw: <b>{my_top['option']}</b> "
                f"(equilibrium weight {my_top['equilibrium_weight']:.0%})", kind="success")
        my_mixers = my_report[my_report["equilibrium_weight"] > 0]
        if len(my_mixers) > 1:
            callout("Equilibrium MIXES between throw options here -- rotate roughly:", kind="warning")
            display_mixers(my_mixers)

        subheader(f"Prediction for {self.opp_team}'s best throw (vs your shield {self.my_shield})")
        display_report(opp_report)
        opp_top = opp_report.iloc[0]
        callout(f"Likely pick if they play rationally: <b>{opp_top['option']}</b> "
                f"(equilibrium weight {opp_top['equilibrium_weight']:.0%}). Confirm once "
                f"revealed -- real opponents don't always play optimally.", kind="info")
        opp_mixers = opp_report[opp_report["equilibrium_weight"] > 0]
        if len(opp_mixers) > 1:
            callout("Their equilibrium also mixes -- don't be surprised by any of:", kind="warning")
            display_mixers(opp_mixers)

        return my_top["option"], opp_top["option"]

    def lock_throws(self, my_throw: Sequence[str], opp_throw: Sequence[str]):
        """Call once both teams' throws are revealed face-up. Immediately
        runs the stage-3 (espada) recommendation, since that's next."""
        if self.my_shield is None:
            raise RuntimeError("Call lock_shields(...) first.")
        remaining_mine = [p for p in self.my_players if p != self.my_shield]
        remaining_theirs = [p for p in self.opp_players if p != self.opp_shield]
        my_throw, opp_throw = tuple(my_throw), tuple(opp_throw)
        self._validate_pair(my_throw, remaining_mine, "my_throw")
        self._validate_pair(opp_throw, remaining_theirs, "opp_throw")
        self.my_throw, self.opp_throw = my_throw, opp_throw
        callout(f"Throws locked &mdash; {self.my_team}: <b>{my_throw}</b> "
                f"| {self.opp_team}: <b>{opp_throw}</b>", kind="info")
        return self.recommend_espada()

    # ---- Stage 3: espada --------------------------------------------------
    def recommend_espada(self):
        """Call once both throws are known: which of the 2 opponents thrown
        at YOUR shield should your shield fight? This accounts for the
        resulting descarte-vs-descarte match too (whichever candidate you
        reject also determines who plays who in that match), so it's
        solved as a small 2x2 game rather than just picking your best
        individual duel -- occasionally that means the equilibrium
        genuinely mixes between your two options, same as the throw stage
        can. Also predicts (but cannot guarantee) the opponent's pick."""
        if self.my_throw is None or self.opp_throw is None:
            raise RuntimeError("Call lock_throws(...) first.")
        s1, s2 = self._to_team1_team2(self.my_shield, self.opp_shield)
        t1, t2 = self._to_team1_team2(self.my_throw, self.opp_throw)

        my_report = espada_decision_report(
            s1, s2, t1, t2, self.model.team1_players, self.model.team2_players,
            self.escudo_df, self.espada_df, self.descarte_df, perspective=self.my_team)
        opp_report = espada_decision_report(
            s1, s2, t1, t2, self.model.team1_players, self.model.team2_players,
            self.escudo_df, self.espada_df, self.descarte_df, perspective=self.opp_team)
        self._espada_reports = (my_report, opp_report)

        section_header(f"STAGE 3 &middot; ESPADA &mdash; your shield {self.my_shield} must pick "
                        f"one of the players thrown at it, {self.opp_throw}")
        callout("Accounts for the resulting descarte-vs-descarte match, not just your "
                "shield's own duel.", kind="info")
        display_report(my_report)
        insight_list(insight_sentences(my_report))
        my_top = my_report.iloc[0]
        callout(f"Recommended pick: <b>{my_top['option']}</b> "
                f"(equilibrium weight {my_top['equilibrium_weight']:.0%})", kind="success")
        my_mixers = my_report[my_report["equilibrium_weight"] > 0]
        if len(my_mixers) > 1:
            callout("Equilibrium MIXES between your options here -- rotate roughly:", kind="warning")
            display_mixers(my_mixers)

        subheader(f"Prediction for {self.opp_team}'s shield {self.opp_shield} "
                  f"(choosing among your thrown players {self.my_throw})")
        opp_top = opp_report.iloc[0]
        callout(f"Likely pick if they play rationally: <b>{opp_top['option']}</b> "
                f"(equilibrium weight {opp_top['equilibrium_weight']:.0%}). Confirm once "
                f"revealed -- real opponents don't always play optimally.", kind="info")

        return my_top["option"], opp_top["option"]

    # ---- Resolution -------------------------------------------------------
    def finalize(self, my_espada_pick: Optional[str] = None,
                 opp_espada_pick: Optional[str] = None):
        """Call once you know the ACTUAL espada picks (your own choice, and
        the opponent's real revealed choice). Leave a value as None to fall
        back to the model's top recommendation/prediction for it."""
        if self._espada_reports is None:
            self.recommend_espada()
        my_report, opp_report = self._espada_reports
        s1, s2 = self._to_team1_team2(self.my_shield, self.opp_shield)
        t1, t2 = self._to_team1_team2(self.my_throw, self.opp_throw)

        my_pick = my_espada_pick or my_report.iloc[0]["option"]
        opp_pick = opp_espada_pick or opp_report.iloc[0]["option"]
        j_star, i_star = (my_pick, opp_pick) if self.my_team == "team1" else (opp_pick, my_pick)

        total_team1, matches = score_combination(
            s1, s2, t1, t2, j_star, i_star,
            self.model.team1_players, self.model.team2_players,
            self.escudo_df, self.espada_df, self.descarte_df)

        my_total = total_team1 if self.my_team == "team1" else 80.0 - total_team1
        opp_total = 80.0 - my_total

        display_final_pairings(matches, self.my_team, my_total, opp_total)

        self.result = {"matches": matches, "my_total": my_total, "opp_total": opp_total}
        return self.result


def _prompt(prompt_text: str, default) -> str:
    """input() wrapper: shows the default value in the prompt and falls
    back to it if you just press Enter with nothing typed."""
    raw = input(f"{prompt_text} [Enter for default: {default}]: ").strip()
    return raw if raw else default


def _prompt_pair(prompt_text: str, default_pair) -> Tuple[str, str]:
    """Same as _prompt, but for the two-name throw fields: pressing Enter
    accepts the default pair as-is; typing anything is parsed as two
    comma-separated names."""
    raw = input(f"{prompt_text} [Enter for default: {default_pair[0]}, {default_pair[1]}]: ").strip()
    if not raw:
        return tuple(default_pair)
    return tuple(x.strip() for x in raw.split(","))


def run_live_cli(model: MatchModelResult, escudo_df: pd.DataFrame,
                  espada_df: pd.DataFrame, descarte_df: pd.DataFrame,
                  my_team: str = "team1") -> LivePairingSession:
    """Fully interactive version -- prompts you for each real event as it
    happens and prints a recommendation after every input. Run this cell
    live during the actual pairing negotiation. Every prompt can be left
    blank (just press Enter) to accept the model's top recommendation or
    prediction for that field."""
    session = LivePairingSession(model, escudo_df, espada_df, descarte_df, my_team=my_team)

    my_shield_default, opp_shield_default = session.recommend_shield()
    my_shield = _prompt(f"\nEnter YOUR chosen shield {session.my_players}", my_shield_default)
    opp_shield = _prompt(f"Enter OPPONENT's revealed shield {session.opp_players}", opp_shield_default)

    my_throw_default, opp_throw_default = session.lock_shields(my_shield, opp_shield)   # auto-prints throw recommendation
    my_throw = _prompt_pair("\nEnter YOUR actual throw as two names, comma-separated", my_throw_default)
    opp_throw = _prompt_pair("Enter OPPONENT's revealed throw as two names, comma-separated", opp_throw_default)

    my_pick_default, opp_pick_default = session.lock_throws(my_throw, opp_throw)      # auto-prints espada recommendation
    my_pick = _prompt("\nWhich opponent did YOUR shield actually pick?", my_pick_default)
    opp_pick = _prompt("Which of your players did the OPPONENT's shield actually pick?", opp_pick_default)

    session.finalize(my_espada_pick=my_pick, opp_espada_pick=opp_pick)
    return session

# Executable

In [ ]:
model = build_model(escudo_df, espada_df, descarte_df)
run_live_cli(model, escudo_df, espada_df, descarte_df, my_team="team1")

option,vs_equilibrium_opponent,worst_case,best_case,spread,equilibrium_weight,gap_to_best
Astra,45.44,44.91,46.00,1.09,50%,+0.00
Death Guard,45.44,44.88,47.00,2.12,50%,+0.00
Marines,43.28,35.00,43.45,8.45,0%,-2.16
Necron,40.84,34.67,41.20,6.53,0%,-4.61


option,equilibrium_weight
Astra,50%
Death Guard,50%


option,vs_equilibrium_opponent,worst_case,best_case,spread,equilibrium_weight,gap_to_best
IK,34.56,34.00,38.80,4.80,31%,+0.00
Marines,34.56,34.30,39.33,5.02,69%,+0.00
CSM,34.55,34.00,40.32,6.32,0%,-0.01
Necron,34.01,33.00,45.33,12.33,0%,-0.55


option,equilibrium_weight
IK,31%
Marines,69%



Enter YOUR chosen shield ['Death Guard', 'Necron', 'Astra', 'Marines'] [Enter for default: Astra]: 
Enter OPPONENT's revealed shield ['Marines', 'CSM', 'IK', 'Necron'] [Enter for default: IK]: 


...

option,equilibrium_weight
"('CSM', 'Necron')",91%
"('Marines', 'Necron')",9%



Enter YOUR actual throw as two names, comma-separated [Enter for default: Necron, Marines]: 
Enter OPPONENT's revealed throw as two names, comma-separated [Enter for default: CSM, Necron]: 


option,vs_equilibrium_opponent,worst_case,best_case,spread,equilibrium_weight,gap_to_best
CSM,45.00,45.00,48.00,3.00,100%,+0.00
Necron,26.00,26.00,43.00,17.00,0%,-19.00



Which opponent did YOUR shield actually pick? [Enter for default: CSM]: 
Which of your players did the OPPONENT's shield actually pick? [Enter for default: Marines]: 


matchup type,you,opponent,your score / 20
escudo,Astra,CSM,20
espada,Marines,IK,6
descarte,Necron,Necron,16
descarte,Death Guard,Marines,3
